# ThinkNCollab Multi-Language ASR Training Pipeline v2.0

**Supported Languages**: Hindi | Hinglish | Indian English | Bengali-Hindi | Tamil-Hindi | Rajasthani-Hindi

**Model**: WhisperSmallHinglish (206.7M Parameters) — Custom Encoder-Decoder PyTorch Architecture

**Pipeline Steps**:
1. Install packages & verify GPU
2. Load multi-language datasets (AI4Bharat, CommonVoice, FLEURS, MUCS)
3. Train SentencePiece BPE tokenizer (4096 vocab) — saved as `hinglish_bpe.model` + `vocab.json`
4. Build & train WhisperSmallHinglish model with CrossEntropy loss + mixed precision (fp16)
5. Save model state dict as `whisper_small_hinglish_v2.pt` + tokenizer vocab JSON
6. Quick inference test on each language variant

In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
!pip install -q torch torchaudio librosa sentencepiece datasets jiwer numpy scipy tqdm

In [ ]:
# ── Cell 2: GPU Verification ──────────────────────────────────────────────────
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch   : {torch.__version__}")
print(f"Device    : {device}")

if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU found. Enable GPU Accelerator in Kaggle Settings > Accelerator > GPU T4.")

In [ ]:
# ── Cell 3: Multi-Language Dataset Loading ────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Audio

SAMPLE_RATE = 16000
datasets_loaded = {}

def safe_load(name, config, split="train", streaming=True):
    try:
        ds = load_dataset(name, config, split=split, streaming=streaming, trust_remote_code=True)
        print(f"[OK] {name} ({config}) loaded")
        return ds
    except Exception as e:
        print(f"[SKIP] {name} ({config}): {e}")
        return None

# Hindi (Devanagari)
datasets_loaded["hindi_kathbath"]     = safe_load("ai4bharat/kathbath", "hindi")
datasets_loaded["hindi_cv"]           = safe_load("mozilla-foundation/common_voice_11_0", "hi")
datasets_loaded["hindi_fleurs"]       = safe_load("google/fleurs", "hi_in")

# Hinglish / Indian English
datasets_loaded["indian_english"]     = safe_load("google/fleurs", "en_in")
datasets_loaded["mucs_meeting"]       = safe_load("ai4bharat/MUCS", "hi")

# Bengali-Hindi (Bengali speech, Hindi transliteration labels)
datasets_loaded["bengali"]            = safe_load("mozilla-foundation/common_voice_11_0", "bn")
datasets_loaded["bengali_fleurs"]     = safe_load("google/fleurs", "bn_in")

# Tamil-Hindi
datasets_loaded["tamil"]              = safe_load("mozilla-foundation/common_voice_11_0", "ta")
datasets_loaded["tamil_fleurs"]       = safe_load("google/fleurs", "ta_in")

# Rajasthani / Bhojpuri adjacent (Kathbath + CommonVoice Dialect sets)
datasets_loaded["rajasthani_cv"]      = safe_load("mozilla-foundation/common_voice_11_0", "raj")

print(f"\nLoaded {sum(1 for v in datasets_loaded.values() if v is not None)} datasets successfully.")

In [ ]:
# ── Cell 4: Extract Transcripts from All Datasets ─────────────────────────────
import json

MAX_SAMPLES_PER_DS = 50000  # Limit per dataset to fit in Kaggle session

all_transcripts = []

for ds_name, ds in datasets_loaded.items():
    if ds is None:
        continue
    count = 0
    try:
        for item in ds:
            text = item.get("sentence") or item.get("transcription") or item.get("text", "")
            if text and text.strip():
                all_transcripts.append(text.strip())
                count += 1
            if count >= MAX_SAMPLES_PER_DS:
                break
        print(f"[{ds_name}] Collected {count} transcripts")
    except Exception as e:
        print(f"[{ds_name}] Error collecting transcripts: {e}")

print(f"\nTotal transcripts collected: {len(all_transcripts)}")

# Save corpus file for SentencePiece training
os.makedirs("/kaggle/working", exist_ok=True)
corpus_path = "/kaggle/working/tnc_corpus.txt"
with open(corpus_path, "w", encoding="utf-8") as f:
    for t in all_transcripts:
        f.write(t + "\n")
print(f"Corpus saved to '{corpus_path}'")

In [ ]:
# ── Cell 5: Train SentencePiece BPE Tokenizer (4096 Vocab) ───────────────────
import sentencepiece as spm
import json

SPM_MODEL_PREFIX = "/kaggle/working/tnc_tokenizer"
VOCAB_SIZE = 4096

spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix=SPM_MODEL_PREFIX,
    vocab_size=VOCAB_SIZE,
    character_coverage=0.9998,    # Covers Devanagari + Latin + Bengali + Tamil
    model_type="bpe",
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3,
    byte_fallback=True,           # Handles unseen Unicode characters (Tamil, Bengali)
    split_digits=True,
    add_dummy_prefix=False,
    normalization_rule_name="nmt_nfkc_cf"
)

# Load trained tokenizer
sp = spm.SentencePieceProcessor()
sp.load(f"{SPM_MODEL_PREFIX}.model")

# Export vocab as JSON for local inference
vocab_json = {}
for i in range(sp.get_piece_size()):
    piece = sp.id_to_piece(i)
    vocab_json[piece] = i

vocab_export_path = "/kaggle/working/tnc_vocab.json"
with open(vocab_export_path, "w", encoding="utf-8") as f:
    json.dump({"vocab_size": len(vocab_json), "special_tokens": ["<pad>", "<s>", "</s>", "<unk>"], "tokens": vocab_json}, f, ensure_ascii=False, indent=2)

print(f"SentencePiece BPE Tokenizer trained: {sp.get_piece_size()} tokens")
print(f"Vocab JSON saved: '{vocab_export_path}'")

# Test tokenizer
test_phrases = [
    "hello am i audible",
    "आज की मीटिंग शुरू हो चुकी है।",
    "Aaj ki meeting start ho chuki hai.",
    "আজকের মিটিং শুরু হয়েছে।",
    "இன்றைய கூட்டம் தொடங்கிவிட்டது.",
    "आज री बैठक शुरू हो ग्यी है।"
]
for phrase in test_phrases:
    tokens = sp.encode(phrase)
    decoded = sp.decode(tokens)
    print(f"  Input  : {phrase}")
    print(f"  Tokens : {tokens[:10]}")
    print(f"  Decoded: {decoded}")
    print()

In [ ]:
# ── Cell 6: Model Architecture (WhisperSmallHinglish) ────────────────────────
import torch
import torch.nn as nn

class WhisperSmallHinglish(nn.Module):
    def __init__(self, vocab_size=4096, n_mels=80, d_model=768, n_heads=12,
                 enc_layers=12, dec_layers=12, ff_dim=3072, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model

        # Audio CNN Stem — converts 80-channel log-mel to 768-dim feature sequence
        self.stem = nn.Sequential(
            nn.Conv1d(n_mels, d_model, kernel_size=3, padding=1),
            nn.SiLU(),
            nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1),
            nn.SiLU()
        )

        # Encoder — 12 Transformer Encoder Blocks
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
                dropout=dropout, batch_first=True, norm_first=True
            ),
            num_layers=enc_layers
        )

        # Token Embedding for Decoder input
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Decoder — 12 Transformer Decoder Blocks with Cross-Attention
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
                dropout=dropout, batch_first=True, norm_first=True
            ),
            num_layers=dec_layers
        )

        # Output projection to vocabulary
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: embedding & output projection share weights
        self.head.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Conv1d)):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def encode(self, mel):
        x = self.stem(mel).permute(0, 2, 1)
        return self.encoder(x)

    def decode(self, tokens, memory, tgt_mask=None, tgt_key_padding_mask=None):
        tgt = self.embedding(tokens)
        out = self.decoder(tgt, memory, tgt_mask=tgt_mask,
                           tgt_key_padding_mask=tgt_key_padding_mask)
        return self.head(out)

    def forward(self, mel, tokens):
        memory = self.encode(mel)
        T = tokens.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=mel.device)
        return self.decode(tokens, memory, tgt_mask=causal_mask)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WhisperSmallHinglish(vocab_size=VOCAB_SIZE).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: WhisperSmallHinglish")
print(f"Total Parameters    : {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Device              : {device}")

In [ ]:
# ── Cell 7: Audio Feature Extraction (Log-Mel Spectrogram) ───────────────────
import numpy as np
import librosa

N_MELS    = 80
N_FFT     = 400
HOP_LEN   = 160
MAX_FRAMES = 3000   # 30 seconds at 16kHz / 160 hop

def audio_to_log_mel(audio_array, sr=16000):
    """Converts raw waveform to normalised 80-channel log-mel spectrogram."""
    if isinstance(audio_array, dict):
        audio_array = np.array(audio_array["array"], dtype=np.float32)
        sr = audio_array if isinstance(audio_array, int) else sr

    y = np.array(audio_array, dtype=np.float32)

    # Normalise loudness
    if y.max() > 1.0:
        y = y / 32768.0

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT,
                                          hop_length=HOP_LEN, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    # Normalise to [0, 1]
    log_mel = (log_mel + 80.0) / 80.0
    log_mel = np.clip(log_mel, 0.0, 1.0)

    # Pad or truncate to MAX_FRAMES
    if log_mel.shape[1] < MAX_FRAMES:
        log_mel = np.pad(log_mel, ((0,0),(0, MAX_FRAMES - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_FRAMES]

    return log_mel.astype(np.float32)

print("Audio feature extractor ready.")
test_mel = audio_to_log_mel(np.zeros(16000*3))
print(f"Test mel shape: {test_mel.shape}  (n_mels x frames)")

In [ ]:
# ── Cell 8: Dataset Loader for Training ───────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader

MAX_TARGET_LEN = 128  # Max token length per transcript

class TNCMultiLangDataset(Dataset):
    def __init__(self, dataset_sources, tokenizer, max_audio_secs=30.0, max_samples=None):
        self.samples = []
        self.tokenizer = tokenizer
        count = 0

        for ds_name, ds in dataset_sources.items():
            if ds is None: continue
            try:
                for item in ds:
                    audio = item.get("audio") or item.get("path", None)
                    text  = item.get("sentence") or item.get("transcription") or item.get("text", "")
                    if not audio or not text or not text.strip():
                        continue
                    self.samples.append((audio, text.strip()))
                    count += 1
                    if max_samples and count >= max_samples:
                        break
            except Exception:
                continue

        print(f"TNCMultiLangDataset: {len(self.samples)} samples loaded")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        audio, text = self.samples[idx]

        if isinstance(audio, dict):
            waveform = np.array(audio["array"], dtype=np.float32)
            sr = audio.get("sampling_rate", 16000)
            if sr != 16000:
                waveform = librosa.resample(waveform, orig_sr=sr, target_sr=16000)
        else:
            waveform, _ = librosa.load(str(audio), sr=16000, mono=True)

        mel = audio_to_log_mel(waveform)
        mel_tensor = torch.tensor(mel, dtype=torch.float32)

        # Tokenize text
        token_ids = [1] + self.tokenizer.encode(text)[:MAX_TARGET_LEN - 2] + [2]  # <s> ... </s>
        if len(token_ids) < MAX_TARGET_LEN:
            token_ids += [0] * (MAX_TARGET_LEN - len(token_ids))  # pad

        token_tensor = torch.tensor(token_ids, dtype=torch.long)
        return mel_tensor, token_tensor

print("Dataset class ready.")

In [ ]:
# ── Cell 9: Training Loop (Mixed-Precision fp16 + Gradient Accumulation) ─────
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# ─── Hyperparameters ────
BATCH_SIZE        = 8
GRAD_ACCUM_STEPS  = 4     # Effective batch = 32
LEARNING_RATE     = 3e-4
WARMUP_STEPS      = 500
MAX_EPOCHS        = 10
MAX_TRAIN_SAMPLES = 200000  # Adjust based on available data

# Build dataset
train_dataset = TNCMultiLangDataset(datasets_loaded, sp, max_samples=MAX_TRAIN_SAMPLES)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01, betas=(0.9, 0.98))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS * len(train_loader))
scaler    = GradScaler()
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore pad token

model.train()
best_loss = float("inf")

for epoch in range(MAX_EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}")
    for step, (mel_batch, token_batch) in enumerate(pbar):
        mel_batch   = mel_batch.to(device)
        token_batch = token_batch.to(device)

        # Teacher forcing: input = tokens[:-1], target = tokens[1:]
        dec_input  = token_batch[:, :-1]
        dec_target = token_batch[:, 1:]

        with autocast():
            logits = model(mel_batch, dec_input)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), dec_target.reshape(-1))
            loss   = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRAD_ACCUM_STEPS
        pbar.set_postfix({"loss": f"{epoch_loss / (step+1):.4f}",
                          "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} complete. Avg Loss: {avg_loss:.4f}")

    # Save checkpoint after every epoch
    ckpt_path = f"/kaggle/working/tnc_checkpoint_epoch_{epoch+1}.pt"
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss,
        "vocab_size": VOCAB_SIZE
    }, ckpt_path)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "/kaggle/working/whisper_small_hinglish_v2_best.pt")
        print(f"  Best model saved (loss: {best_loss:.4f})")

print("Training complete!")

In [ ]:
# ── Cell 10: Save Final Model + Tokenizer (Local Inference Format) ────────────
import os, json, shutil

SAVE_DIR = "/kaggle/working/tnc_model_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save final model state dict
torch.save(model.state_dict(), f"{SAVE_DIR}/whisper_small_hinglish_v2.pt")

# 2. Save SentencePiece tokenizer model
shutil.copy(f"{SPM_MODEL_PREFIX}.model", f"{SAVE_DIR}/tnc_tokenizer.model")
shutil.copy(f"{SPM_MODEL_PREFIX}.vocab", f"{SAVE_DIR}/tnc_tokenizer.vocab")

# 3. Save vocabulary JSON (for local Python inference without sentencepiece installed)
shutil.copy(vocab_export_path, f"{SAVE_DIR}/tnc_vocab.json")

# 4. Save model config JSON
config = {
    "model_name": "WhisperSmallHinglish",
    "version": "2.0",
    "vocab_size": VOCAB_SIZE,
    "n_mels": N_MELS,
    "d_model": 768,
    "n_heads": 12,
    "enc_layers": 12,
    "dec_layers": 12,
    "ff_dim": 3072,
    "max_frames": MAX_FRAMES,
    "sample_rate": 16000,
    "supported_languages": ["hindi", "hinglish", "indian_english", "bengali_hindi", "tamil_hindi", "rajasthani_hindi"],
    "total_parameters": sum(p.numel() for p in model.parameters())
}
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"All model files saved to '{SAVE_DIR}'")
print(f"Files saved:")
for fname in os.listdir(SAVE_DIR):
    size_mb = os.path.getsize(f"{SAVE_DIR}/{fname}") / 1024 / 1024
    print(f"  {fname}  ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 11: Inference Test on All 6 Language Variants ───────────────────────
import numpy as np

model.eval()

def infer(waveform, sr=16000, max_new_tokens=50, language="hindi"):
    log_mel = audio_to_log_mel(waveform)
    mel_t   = torch.tensor(log_mel, dtype=torch.float32).unsqueeze(0).to(device)

    generated = [1]  # <s>
    with torch.no_grad():
        memory = model.encode(mel_t)
        for _ in range(max_new_tokens):
            dec_in = torch.tensor([generated], dtype=torch.long, device=device)
            T = dec_in.size(1)
            mask = nn.Transformer.generate_square_subsequent_mask(T, device=device)
            logits = model.decode(dec_in, memory, tgt_mask=mask)
            next_tok = int(logits[0, -1].argmax().item())
            if next_tok == 2: break  # </s>
            generated.append(next_tok)

    text = sp.decode(generated[1:])  # Skip <s>
    return text

# Test with synthetic audio (sine wave)
t = np.linspace(0, 3.0, 16000 * 3)
test_audio = (0.3 * np.sin(2 * np.pi * 300 * t) + 0.2 * np.sin(2 * np.pi * 500 * t)).astype(np.float32)

print("Inference test results:")
for lang in ["hindi", "hinglish", "indian_english", "bengali_hindi", "tamil_hindi", "rajasthani_hindi"]:
    result = infer(test_audio, language=lang)
    print(f"  [{lang:20s}]: {result if result.strip() else '(No speech detected)'}")

In [ ]:
# ── Cell 12: Download Instructions ───────────────────────────────────────────
print("="*60)
print("MODEL DOWNLOAD INSTRUCTIONS")
print("="*60)
print()
print("Download these files from Kaggle Output and place in TNC_TRANSCRIPT/:")
print()
print("  tnc_model_v2/whisper_small_hinglish_v2.pt  -> checkpoints/")
print("  tnc_model_v2/tnc_vocab.json               -> data/hinglish_bpe.json")
print("  tnc_model_v2/tnc_tokenizer.model          -> data/tnc_tokenizer.model")
print("  tnc_model_v2/config.json                  -> checkpoints/config.json")
print()
print("Then update load_trained_model.py checkpoint path to:")
print("  CHECKPOINT_PATH = 'checkpoints/whisper_small_hinglish_v2.pt'")
print()
print(f"Model Size: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Checkpoint: {os.path.getsize('/kaggle/working/tnc_model_v2/whisper_small_hinglish_v2.pt') / 1024 / 1024:.1f} MB")